In [1]:
import requests
from bs4 import BeautifulSoup

In [3]:
# URL of the webpage
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

# Define headers with a user-agent to mimic a web browser

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

In [11]:
# Send a GET request to the URL with headers
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content of the webpage
    soup = BeautifulSoup(response.content, "html.parser")

    # Find the download link within the webpage
    download_link = soup.find("a", {"rel": "nofollow", "href": "?download=csv"})

    # # If the download link is found
    if download_link:
    #     # Extract the href attribute which contains the actual download link
        download_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'+download_link["href"]

    #     # Download the CSV file using the obtained download URL
        download_response = requests.get(download_url, headers=headers)

    #     # Check if the download request was successful
        if download_response.status_code == 200:
            # Save the content of the response to a local file
            with open("global_stocks.csv", "wb") as f:
                f.write(download_response.content)
            print("CSV file downloaded successfully.")
        else:
            print("Failed to download the CSV file.")
    # else:
    #     print("Download link not found on the webpage.")
else:
    print("Failed to retrieve data from the webpage.")

None


<h2> Question 1 </h2>

In [13]:
import pandas as pd

In [14]:
# Download the Wikipedia page
response = requests.get(url, headers=headers)
response.raise_for_status()

# The first table contains the current S&P 500 companies
sp500 = pd.read_html(response.text)[0]

# Create a DataFrame with ticker, company name, and addition date
companies = sp500[["Symbol", "Security", "Date added"]].copy()

# Convert addition dates to datetime and extract the year
companies["Date added"] = pd.to_datetime(
    companies["Date added"],
    errors="coerce"
)
companies["Year added"] = companies["Date added"].dt.year.astype("Int64")

print(companies.head())

  Symbol             Security Date added  Year added
0    MMM                   3M 1957-03-04        1957
1    AOS          A. O. Smith 2017-07-26        2017
2    ABT  Abbott Laboratories 1957-03-04        1957
3   ABBV               AbbVie 2012-12-31        2012
4    ACN            Accenture 2011-07-06        2011


/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_64709/2880275507.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(response.text)[0]


In [15]:
additions_by_year = (
    companies.loc[companies["Year added"] >= 2020]
    .groupby("Year added")
    .size()
    .rename("Number of additions")
    .reset_index()
    .sort_values("Year added")
)

print(additions_by_year)

   Year added  Number of additions
0        2020                   10
1        2021                   10
2        2022                   15
3        2023                   15
4        2024                   16
5        2025                   18
6        2026                   13


<h2> Question 2 </h2>

In [17]:
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

In [23]:
indexes = {
    "S&P 500": "^GSPC",
    "Shanghai Composite": "000001.SS",
    "Hang Seng": "^HSI",
    "ASX 200": "^AXJO",
    "Nifty 50": "^NSEI",
    "TSX Composite": "^GSPTSE",
    "DAX": "^GDAXI",
    "FTSE 100": "^FTSE",
    "Nikkei 225": "^N225",
    "IPC Mexico": "^MXX",
    "Ibovespa": "^BVSP"
}

In [22]:
end = date(year=2026, month=8, day=22)
print(f'Year = {end.year}; month= {end.month}; day={end.day}')

start = date(year=2026, month=1, day=1)

Year = 2026; month= 8; day=22


In [24]:
ticker_obj = yf.Ticker("^GSPC")

data = ticker_obj.history(
    start=start,
    end=end,
    interval="1d"
)

first_close = data["Close"].iloc[0]
last_close = data["Close"].iloc[-1]

ytd_return = last_close / first_close - 1

print(first_close)
print(last_close)
print(ytd_return)

6858.47021484375
7674.3701171875
0.11896237452163927


In [25]:
results = {}

for name, ticker in indexes.items():
    data = yf.Ticker(ticker).history(
        start=start,
        end=end,
        interval="1d"
    )

    first_close = data["Close"].iloc[0]
    last_close = data["Close"].iloc[-1]

    ytd_return = last_close / first_close - 1

    results[name] = ytd_return

In [28]:
results_df = pd.DataFrame(
    list(results.items()),
    columns=["Index", "YTD_Return"]
)

results_df["YTD_Return_Pct"] = results_df["YTD_Return"] * 100

results_df.sort_values("YTD_Return", ascending=False)

,Index,YTD_Return,YTD_Return_Pct
8,Nikkei 225,0.273641,27.364060
5,TSX Composite,0.148566,14.856630
0,S&P 500,0.118962,11.896237
7,FTSE 100,0.086975,8.697531
10,Ibovespa,0.065361,6.536106
6,DAX,0.065088,6.508817
3,ASX 200,0.037936,3.793632
9,IPC Mexico,0.024755,2.475501
2,Hang Seng,-0.012492,-1.249160
1,Shanghai Composite,-0.029382,-2.938152


<h2> Question 3 </h2>

In [29]:
sp500 = yf.download("^GSPC", start="1950-01-01", interval="1d")

sp500.head()

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
1950-01-03,16.66,16.66,16.66,16.66,1260000
1950-01-04,16.85,16.85,16.85,16.85,1890000
1950-01-05,16.93,16.93,16.93,16.93,2550000
1950-01-06,16.98,16.98,16.98,16.98,2010000
1950-01-09,17.08,17.08,17.08,17.08,2520000


In [30]:
sp500.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

In [50]:
sp500.columns = sp500.columns.get_level_values(0)

In [53]:
sp500.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Running_Max', 'Is_ATH'], dtype='object', name='Price')

In [51]:
sp500["Running_Max"] = sp500["Close"].cummax()
sp500.head()

Price,Close,High,Low,Open,Volume,Running_Max,Is_ATH
Date,,,,,,,
1950-01-03,16.66,16.66,16.66,16.66,1260000,16.66,False
1950-01-04,16.85,16.85,16.85,16.85,1890000,16.85,True
1950-01-05,16.93,16.93,16.93,16.93,2550000,16.93,True
1950-01-06,16.98,16.98,16.98,16.98,2010000,16.98,True
1950-01-09,17.08,17.08,17.08,17.08,2520000,17.08,True


In [54]:

sp500["Is_ATH"] = (
    sp500["Close"] >
    sp500["Running_Max"].shift(1)
)

ath = sp500[sp500["Is_ATH"]]
ath_dates = ath.index

In [55]:
corrections = []

for i in range(len(ath_dates) - 1):
    high_date = ath_dates[i]
    next_high_date = ath_dates[i + 1]

    period = sp500.loc[high_date:next_high_date]

    high_price = sp500.loc[high_date, "Close"]

    low_price = period["Close"].min()
    low_date = period["Close"].idxmin()

    drawdown = (high_price - low_price) / high_price * 100

    duration = (low_date - high_date).days

    corrections.append({
        "High_Date": high_date,
        "Low_Date": low_date,
        "High": high_price,
        "Low": low_price,
        "Drawdown_Pct": drawdown,
        "Duration_Days": duration
    })

In [56]:
corrections_df = pd.DataFrame(corrections)
corrections_df.head()

,High_Date,Low_Date,High,Low,Drawdown_Pct,Duration_Days
0,1950-01-04,1950-01-04,16.85,16.850000,0.000000,0
1,1950-01-05,1950-01-05,16.93,16.930000,0.000000,0
2,1950-01-06,1950-01-06,16.98,16.980000,0.000000,0
3,1950-01-09,1950-01-10,17.08,17.030001,0.292736,1
4,1950-01-11,1950-01-13,17.09,16.670000,2.457578,2


In [43]:
low_date = period["Close"].idxmin()
low_date = low_date.iloc[0]

In [57]:
significant = corrections_df[
    corrections_df["Drawdown_Pct"] >= 5
]

In [58]:
significant.head()

,High_Date,Low_Date,High,Low,Drawdown_Pct,Duration_Days
34,1950-06-12,1950-07-17,19.400000,16.680000,14.020615,35
41,1950-11-24,1950-12-04,20.320000,19.000000,6.496062,10
61,1951-05-03,1951-06-29,22.809999,20.959999,8.110480,57
75,1951-10-15,1951-11-23,23.850000,22.400000,6.079668,39
83,1952-01-22,1952-02-20,24.660000,23.090000,6.366584,29


In [59]:
significant.sort_values(
    "Drawdown_Pct",
    ascending=False
).head(10)

,High_Date,Low_Date,High,Low,Drawdown_Pct,Duration_Days
1039,2007-10-09,2009-03-09,1565.150024,676.530029,56.775388,517
1030,2000-03-24,2002-10-09,1527.459961,776.760010,49.146948,929
527,1973-01-11,1974-10-03,120.239998,62.279999,48.203593,630
492,1968-11-29,1970-05-26,108.370003,69.290001,36.061641,543
1294,2020-02-19,2020-03-23,3386.149902,2237.399902,33.924960,33
703,1987-08-25,1987-12-04,336.769989,223.919998,33.509515,101
326,1961-12-12,1962-06-26,72.639999,52.320000,27.973568,196
551,1980-11-28,1982-08-12,140.520004,102.419998,27.113582,622
1385,2022-01-03,2022-10-12,4796.560059,3577.030029,25.425097,282
445,1966-02-09,1966-10-07,94.059998,73.199997,22.177335,240


In [60]:
significant["Drawdown_Pct"].quantile([0.25, 0.50, 0.75])

0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: Drawdown_Pct, dtype: float64

In [61]:
significant["Duration_Days"].quantile([0.25, 0.50, 0.75])

0.25    22.00
0.50    40.50
0.75    86.25
Name: Duration_Days, dtype: float64

In [62]:
median_drawdown = significant["Drawdown_Pct"].median()
median_drawdown

7.986357561745869

In [ ]:
print(f"Median drawdown: {median_drawdown:.2f}%")